# Area Modular V5 — frozen GPU parity
Private baseline; no OSM/BQ re-fetch, no Traffic Counter. Kaggle GPU image; strict frozen scientific/core versions. This is validation, not a production readiness claim.

In [ ]:
import json,subprocess,os,ctypes,glob
from pathlib import Path
report={'schema':'area_cuda_host_probe_v1','device_nodes':glob.glob('/dev/nvidia*'),
        'cuda_visible_devices':os.environ.get('CUDA_VISIBLE_DEVICES')}
try:
 result=subprocess.run(['nvidia-smi'],capture_output=True,text=True,timeout=20)
 report.update(nvidia_smi_code=result.returncode,nvidia_smi_stdout=result.stdout,nvidia_smi_stderr=result.stderr)
except Exception as e:report.update(nvidia_smi_error=type(e).__name__)
try:
 driver=ctypes.CDLL('libcuda.so.1');version=ctypes.c_int()
 report['driver_get_version_code']=driver.cuDriverGetVersion(ctypes.byref(version))
 report['driver_version']=version.value;report['cu_init_code']=driver.cuInit(0)
except Exception as e:report['driver_load_error']=type(e).__name__
Path('/kaggle/working/cuda_host_probe.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
if report.get('cu_init_code')!=0:raise RuntimeError('KAGGLE_CUDA_HOST_NOT_READY: no expensive computation started')


In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys,zipfile
hits=list(Path('/kaggle/input').rglob('baseline.bin'));assert len(hits)==1
BASELINE=hits[0]
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
 return h.hexdigest()
assert sha(BASELINE)=='f7340d3fd913020ddf661dc87674ff204511095f2494bd9b5b0036d7575bbc8c'
wheels=list(BASELINE.parent.glob('engine-wheel.bin'));assert len(wheels)==1
assert sha(wheels[0])=='1f55872c1b1539ff790ac30a2f897a43f7046ace536d3c91f11ca546926b90be'
import shutil
wheel_path=Path('/kaggle/working')/'traffic_location_engines-0.3.0rc1-py3-none-any.whl'
shutil.copyfile(wheels[0],wheel_path)
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel_path)],check=True)
print('Verified frozen baseline and modular wheel')

source_hits=list(BASELINE.parent.glob('source-bundle.bin'));assert len(source_hits)==1
SOURCE_BUNDLE=source_hits[0]
assert sha(SOURCE_BUNDLE)=='72408fe67255a16411885ebbdb36ba2e8dd928f6e091620760ce8b222599a28d'


In [ ]:
from traffic_location.location.area._kernel.location_intelligent_kaggle_esri import setup_dependencies
CORE_PYTHON,OSMIUM=setup_dependencies()
with zipfile.ZipFile(BASELINE) as z:expected=json.loads(z.read('pilot/config/source_software.json'))['packages']
# Pin versions observed at calculation time; full distribution inventories may contain stale duplicates.
names=['numpy','pandas','shapely','pyproj','networkx','folium','osmnx']
subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:',*[n+'=='+expected[n] for n in names]],check=True,timeout=900)
print('Frozen scientific versions:',expected)


In [ ]:
from traffic_location.location.area.application.pipeline import execute
OUTPUT=Path('/kaggle/working/modular_area_validation')
request={'schema_version':'area_execution_request_v2','operation':'compute_gpu_sources',
 'input':{'path':str(SOURCE_BUNDLE),'sha256':sha(SOURCE_BUNDLE)},'core_python':CORE_PYTHON,
 'render':True,'timeout_seconds':3600}
result=execute(request,OUTPUT)
print(json.dumps(result,indent=2))

from traffic_location.location.area.sources.snapshots import MarketSnapshot
from traffic_location.artifacts import read,write
from traffic_location.location.area.domain.comparison import routing_view
import gzip
expected=MarketSnapshot(BASELINE);observed=read(OUTPUT/'native.json.gz')
fields=['rows','summary','stores','excluded_stores','attractor_gate','market_inputs_complete','competitor_groups','routing_clusters']
assert all(observed[k]==expected.native[k] for k in fields),'Full market mismatch'
calc=['graph/graph.json.gz','connectors/connectors.json.gz','normalized/metric_buildings.json','building_results/routing.json.gz','building_results/decisions.json.gz']
with zipfile.ZipFile(BASELINE) as z:
 for name in calc:
  raw=z.read('pilot/'+name);old=json.loads(gzip.decompress(raw) if name.endswith('.gz') else raw)
  new=read(OUTPUT/'computation/pilot'/name)
  assert (routing_view(old)==routing_view(new) if name=='building_results/routing.json.gz' else old==new),name
capacity=read(OUTPUT/'capacity.json')
assert {k:v for k,v in capacity.items() if k not in {'policy_hash','selection_hash'}}=={k:v for k,v in expected.capacity.items() if k not in {'policy_hash','selection_hash'}}
write('/kaggle/working/reference_parity.json',{'schema':'area_source_only_reference_parity_v1','status':'PASS','market_fields':fields,'calculation_fields':calc,'capacity_domain_exact':True,'input_contains_prior_results':False,'source_bundle_sha256':sha(SOURCE_BUNDLE),'golden_sha256':sha(BASELINE),'domain_sha256':result['native_domain_sha256']})
print('SOURCE-ONLY COMPUTATION PARITY: PASS')


In [ ]:
from IPython.display import display,HTML,FileLink
from html import escape
if result['presentation_ref']:
 path=OUTPUT/result['presentation_ref']
 display(HTML('<iframe srcdoc="'+escape(path.read_text(),quote=True)+'" style="width:100%;height:850px;border:0"></iframe>'))
for name in ['result.json','manifest.json','native.json.gz','capacity.json']:
 display(FileLink(str(OUTPUT/name)))
